# Graph Partitioning

In graph theory, you can partition a graph in many ways. This tutorial shows how to partition graphs using three methods:

1. **Community Detection (Label Propagation)** - Groups vertices based on modularity optimization
2. **Edge Betweenness** - Partitions by removing high-betweenness edges
3. **Fiedler Vector / Spectral** - Uses eigenvectors of the Laplacian matrix

**Note**: This notebook uses topologic_fast's native `Graph.Partition()`, `Graph.ByAdjacencyMatrix()`, and `Graph.Reshape()` implementations.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import numpy as np
from collections import defaultdict, deque
from scipy import sparse
from scipy.sparse.linalg import eigsh

## Create a Graph from an Adjacency Matrix

We'll use the high school building layout for demonstration.

In [ ]:
# Adjacency Matrix for a fictional High School
adjacencyMatrix = [[0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
                   [1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0],
                   [0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0],
                   [0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0],
                   [1,1,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
                   [1,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0],
                   [0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0],
                   [0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0],
                   [0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0]]

node_names = [
    "Daycare_Room",
    "Elementary_School",
    "High_School_Wing",
    "Library",
    "Corridor_1",
    "Study_Room",
    "Special_Education_Room",
    "Swimming_Pool",
    "Gymnasium",
    "Outdoor_Space",
    "Sports_Field",
    "Corridor_2",
    "Main_Entrance_Hall",
    "Lobby",
    "Administrative_Office",
    "Principals_Office",
    "Counseling_Office",
    "Corridor_3",
    "Cafeteria",
    "Corridor_4",
    "Kitchen",
    "Corridor_5",
    "Workshop",
    "Corridor_6"]

print(f"Number of nodes: {len(node_names)}")
print(f"Number of edges: {sum(sum(row) for row in adjacencyMatrix) // 2}")

## Build the Graph and Layout

We'll use `Graph.ByAdjacencyMatrix` to create the graph and `Graph.Reshape` for the visualization layout.

In [ ]:
# Create the graph directly from the adjacency matrix using topologic_fast
graph = tf.Graph.ByAdjacencyMatrix(adjacencyMatrix)

print(f"Graph created with {graph.Order()} vertices and {graph.Size()} edges")

# Use Graph.Reshape to get spring layout positions for visualization
positions_dict = graph.Reshape(layout_type="spring2d", iterations=100)

# Convert to numpy array for visualization
positions = np.array([[positions_dict[i][0], positions_dict[i][1]] for i in range(len(node_names))])
print(f"Layout computed using Graph.Reshape with spring2d layout")

## Visualize Original Graph

In [ ]:
def visualize_partition(positions, adj_matrix, node_names, partition, title="Graph Partition"):
    """
    Visualize a graph with vertex coloring based on partition.
    
    Args:
        positions: numpy array of (x, y) coordinates
        adj_matrix: adjacency matrix
        node_names: list of node names
        partition: list of partition labels (integers) for each node
        title: plot title
    """
    fig = go.Figure()
    
    # Color palette for partitions
    colors = ['red', 'green', 'blue', 'cyan', 'magenta', 'yellow', 'orange', 'purple', 'brown', 'pink', 'gray']
    
    # Get unique partitions
    unique_partitions = sorted(set(partition))
    n_partitions = len(unique_partitions)
    
    # Draw edges
    for i in range(len(adj_matrix)):
        for j in range(i+1, len(adj_matrix)):
            if adj_matrix[i][j] == 1:
                # Color edge based on whether it's within or between partitions
                if partition[i] == partition[j]:
                    part_idx = unique_partitions.index(partition[i]) % len(colors)
                    edge_color = colors[part_idx]
                    width = 3
                else:
                    edge_color = 'lightgray'
                    width = 1
                
                fig.add_trace(go.Scatter(
                    x=[positions[i, 0], positions[j, 0]],
                    y=[positions[i, 1], positions[j, 1]],
                    mode='lines',
                    line=dict(color=edge_color, width=width),
                    hoverinfo='skip',
                    showlegend=False
                ))
    
    # Draw vertices by partition
    for p_idx, p in enumerate(unique_partitions):
        p_nodes = [i for i, part in enumerate(partition) if part == p]
        color = colors[p_idx % len(colors)]
        
        fig.add_trace(go.Scatter(
            x=[positions[i, 0] for i in p_nodes],
            y=[positions[i, 1] for i in p_nodes],
            mode='markers+text',
            marker=dict(
                size=18,
                color=color,
                line=dict(color='black', width=2)
            ),
            text=[node_names[i] for i in p_nodes],
            textposition='top center',
            textfont=dict(size=8),
            hovertext=[f"{node_names[i]}\nPartition: {p}" for i in p_nodes],
            hoverinfo='text',
            name=f'Partition {p}',
            showlegend=True
        ))
    
    fig.update_layout(
        title=f'{title}<br><sub>{n_partitions} partitions found</sub>',
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor='x'),
        width=1000,
        height=800,
        legend=dict(x=1.02, y=1),
        hovermode='closest',
        plot_bgcolor='white'
    )
    
    return fig

# Original graph (all in one partition)
original_partition = [0] * len(node_names)
fig = visualize_partition(positions, adjacencyMatrix, node_names, original_partition, "Original Graph")
fig.show()

## Method 1: Community Detection (Label Propagation)

topologic_fast provides `Graph.Partition(method="label_propagation")` for community detection using label propagation, which optimizes for community structure.

In [ ]:
# Compute community partition using topologic_fast's native implementation
# method can be "label_propagation", "spectral", or "edge_betweenness"
community_partition = graph.Partition(method="label_propagation")

# Compute modularity for analysis
def compute_modularity(adj_matrix, partition):
    """
    Compute modularity Q for a given partition.
    Q = (1/2m) * sum_ij [(A_ij - k_i*k_j/2m) * delta(c_i, c_j)]
    """
    n = len(adj_matrix)
    m = sum(sum(row) for row in adj_matrix) / 2
    if m == 0:
        return 0
    
    degrees = [sum(row) for row in adj_matrix]
    Q = 0
    
    for i in range(n):
        for j in range(n):
            if partition[i] == partition[j]:
                Q += adj_matrix[i][j] - (degrees[i] * degrees[j]) / (2 * m)
    
    return Q / (2 * m)

modularity = compute_modularity(adjacencyMatrix, community_partition)

print("Community/Label Propagation Partitioning (computed with topologic_fast):")
print(f"Number of communities: {len(set(community_partition))}")
print(f"Modularity: {modularity:.4f}")
print("\nPartition assignments:")
for p in sorted(set(community_partition)):
    members = [node_names[i] for i, part in enumerate(community_partition) if part == p]
    print(f"  Community {p}: {', '.join(members)}")

In [ ]:
fig = visualize_partition(positions, adjacencyMatrix, node_names, community_partition, "Community/Louvain Partitioning")
fig.show()

## Method 2: Edge Betweenness Partitioning

topologic_fast provides `Graph.Partition(method="edge_betweenness")` which uses the Girvan-Newman algorithm to partition by removing high-betweenness edges.

In [ ]:
# Compute edge betweenness partition using topologic_fast's native implementation
# n_partitions parameter specifies desired number of partitions
betweenness_part = graph.Partition(method="edge_betweenness", n_partitions=4)

print("Edge Betweenness Partitioning (computed with topologic_fast):")
print(f"Number of partitions: {len(set(betweenness_part))}")
print("\nPartition assignments:")
for p in sorted(set(betweenness_part)):
    members = [node_names[i] for i, part in enumerate(betweenness_part) if part == p]
    print(f"  Partition {p}: {', '.join(members)}")

In [ ]:
fig = visualize_partition(positions, adjacencyMatrix, node_names, betweenness_part, "Edge Betweenness Partitioning")
fig.show()

## Method 3: Fiedler Vector / Spectral Partitioning

topologic_fast provides `Graph.Partition(method="spectral")` which uses the Fiedler vector (second smallest eigenvector of the graph Laplacian) for spectral bisection.

In [ ]:
# Compute spectral/Fiedler partition using topologic_fast's native implementation
fiedler_part = graph.Partition(method="spectral")

print("Fiedler Vector / Spectral Partitioning (computed with topologic_fast):")
print(f"Number of partitions: {len(set(fiedler_part))}")
print("\nPartition assignments:")
for p in sorted(set(fiedler_part)):
    members = [node_names[i] for i, part in enumerate(fiedler_part) if part == p]
    print(f"  Partition {p}: {', '.join(members)}")

In [ ]:
fig = visualize_partition(positions, adjacencyMatrix, node_names, fiedler_part, "Fiedler Vector / Spectral Partitioning")
fig.show()

## Comparison of Partitioning Methods

In [ ]:
def count_cut_edges(adj_matrix, partition):
    """Count edges between different partitions."""
    n = len(adj_matrix)
    cut_edges = 0
    for i in range(n):
        for j in range(i+1, n):
            if adj_matrix[i][j] == 1 and partition[i] != partition[j]:
                cut_edges += 1
    return cut_edges

def partition_balance(partition):
    """Compute balance metric (how evenly sized the partitions are)."""
    from collections import Counter
    counts = Counter(partition)
    sizes = list(counts.values())
    mean_size = np.mean(sizes)
    std_size = np.std(sizes)
    return mean_size, std_size, min(sizes), max(sizes)

print("COMPARISON OF PARTITIONING METHODS")
print("=" * 70)

methods = [
    ("Community/Louvain", community_partition),
    ("Edge Betweenness", betweenness_part),
    ("Fiedler/Spectral", fiedler_part)
]

total_edges = sum(sum(row) for row in adjacencyMatrix) // 2

print(f"\nTotal edges in graph: {total_edges}")
print(f"Total nodes: {len(node_names)}")
print()

for method_name, partition in methods:
    n_parts = len(set(partition))
    modularity = compute_modularity(adjacencyMatrix, partition)
    cut_edges = count_cut_edges(adjacencyMatrix, partition)
    mean_size, std_size, min_size, max_size = partition_balance(partition)
    
    print(f"{method_name}:")
    print(f"  Partitions:    {n_parts}")
    print(f"  Modularity:    {modularity:.4f}")
    print(f"  Cut edges:     {cut_edges} ({100*cut_edges/total_edges:.1f}% of total)")
    print(f"  Size range:    {min_size} to {max_size} nodes")
    print(f"  Size std dev:  {std_size:.2f}")
    print()

## Side-by-Side Visualization

In [ ]:
from plotly.subplots import make_subplots

def create_partition_subplot(fig, row, col, positions, adj_matrix, node_names, partition, title):
    """Add a partition visualization to a subplot."""
    colors = ['red', 'green', 'blue', 'cyan', 'magenta', 'yellow', 'orange', 'purple', 'brown', 'pink']
    unique_partitions = sorted(set(partition))
    
    # Draw edges
    for i in range(len(adj_matrix)):
        for j in range(i+1, len(adj_matrix)):
            if adj_matrix[i][j] == 1:
                if partition[i] == partition[j]:
                    part_idx = unique_partitions.index(partition[i]) % len(colors)
                    edge_color = colors[part_idx]
                    width = 2
                else:
                    edge_color = 'lightgray'
                    width = 1
                
                fig.add_trace(go.Scatter(
                    x=[positions[i, 0], positions[j, 0]],
                    y=[positions[i, 1], positions[j, 1]],
                    mode='lines',
                    line=dict(color=edge_color, width=width),
                    showlegend=False
                ), row=row, col=col)
    
    # Draw vertices
    for p_idx, p in enumerate(unique_partitions):
        p_nodes = [i for i, part in enumerate(partition) if part == p]
        color = colors[p_idx % len(colors)]
        
        fig.add_trace(go.Scatter(
            x=[positions[i, 0] for i in p_nodes],
            y=[positions[i, 1] for i in p_nodes],
            mode='markers',
            marker=dict(size=12, color=color, line=dict(color='black', width=1)),
            showlegend=False
        ), row=row, col=col)

# Create subplot figure
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Community/Louvain', 'Edge Betweenness', 'Fiedler/Spectral']
)

create_partition_subplot(fig, 1, 1, positions, adjacencyMatrix, node_names, community_partition, "Community")
create_partition_subplot(fig, 1, 2, positions, adjacencyMatrix, node_names, betweenness_part, "Betweenness")
create_partition_subplot(fig, 1, 3, positions, adjacencyMatrix, node_names, fiedler_part, "Fiedler")

fig.update_layout(
    title='Comparison of Graph Partitioning Methods',
    width=1500,
    height=500,
    showlegend=False
)

for i in range(1, 4):
    fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, row=1, col=i)
    fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, scaleanchor=f'x{i}', row=1, col=i)

fig.show()

## Summary

This notebook demonstrated three graph partitioning methods using topologic_fast:

### 1. Community Detection (Label Propagation)
- Uses label propagation algorithm for community detection
- Good for finding natural groupings
- Number of partitions emerges from the algorithm

### 2. Edge Betweenness (Girvan-Newman)
- Removes high-betweenness edges iteratively
- Good for finding bottlenecks
- Can specify desired number of partitions

### 3. Fiedler Vector (Spectral)
- Uses eigenvectors of graph Laplacian
- Creates balanced partitions
- Mathematically elegant bisection

### topologic_fast Methods Used
- `Graph.ByAdjacencyMatrix(matrix)` - Create graph from adjacency matrix
- `Graph.Partition(method, n_partitions)` - Partition graph using various methods:
  - `method="label_propagation"` - Label propagation community detection
  - `method="edge_betweenness"` - Girvan-Newman algorithm
  - `method="spectral"` - Fiedler vector spectral bisection
- `Graph.Reshape(layout_type, iterations)` - Force-directed spring layout

### Applications:
- Building zone classification
- Network segmentation
- Community detection in social networks
- Load balancing in distributed systems